# Air Quality Prediction using OpenAQ + NASA POWER
# Author: Reesha and Eesha Waqar
# Description: This notebook collects, processes, and prepares data for AQI prediction.

In [1]:
import requests
import pandas as pd
import numpy as np

In [2]:
city = "Lahore"
lat = 31.5204
lon = 74.3587

start_date = "20220101"
end_date = "20231231"

In [3]:
nasa_url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    f"?parameters=T2M,RH2M,WS2M"
    f"&community=RE"
    f"&longitude={lon}"
    f"&latitude={lat}"
    f"&start={start_date}"
    f"&end={end_date}"
    f"&format=JSON"
)

response = requests.get(nasa_url)
data = response.json()

weather = pd.DataFrame(data["properties"]["parameter"])
weather.index = pd.to_datetime(weather.index)
weather = weather.reset_index().rename(columns={"index": "date"})

weather.head()

,date,T2M,RH2M,WS2M
0,2022-01-01,12.42,44.13,1.09
1,2022-01-02,12.92,45.62,1.37
2,2022-01-03,13.75,52.70,1.28
3,2022-01-04,12.59,69.26,1.51
4,2022-01-05,12.47,90.84,0.87


In [4]:
weather.info()
weather.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730 entries, 0 to 729
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    730 non-null    datetime64[ns]
 1   T2M     730 non-null    float64       
 2   RH2M    730 non-null    float64       
 3   WS2M    730 non-null    float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 22.9 KB


,date,T2M,RH2M,WS2M
count,730,730.000000,730.000000,730.000000
mean,2022-12-31 12:00:00,24.684699,55.151301,1.467521
min,2022-01-01 00:00:00,7.540000,9.500000,0.430000
25%,2022-07-02 06:00:00,17.472500,42.505000,1.090000
50%,2022-12-31 12:00:00,26.340000,56.985000,1.365000
75%,2023-07-01 18:00:00,30.515000,70.225000,1.725000
max,2023-12-31 00:00:00,39.890000,91.090000,5.090000
std,NaN,7.874185,18.946857,0.539455


In [6]:
headers = {
    "X-API-Key": "e67f495428ae4db02e150d673eccf5a9dcb75f07aba85f91bfc078a9d995e7f4"
}

locations_url = "https://api.openaq.org/v3/locations"

params = {
    "coordinates": f"{lat},{lon}",
    "radius": 25000,
    "limit": 20
}

response = requests.get(locations_url, params=params, headers=headers)
openaq_locations = response.json()

openaq_locations

{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 20,
  'found': '>20'},
 'results': [{'id': 8664,
   'name': 'US Diplomatic Post: Lahore',
   'locality': 'Lahore',
   'timezone': 'Asia/Karachi',
   'country': {'id': 109, 'code': 'PK', 'name': 'Pakistan'},
   'owner': {'id': 4, 'name': 'Unknown Governmental Organization'},
   'provider': {'id': 265, 'name': 'StateAir Lahore'},
   'isMobile': False,
   'isMonitor': True,
   'instruments': [{'id': 2, 'name': 'Government Monitor'}],
   'sensors': [{'id': 25135,
     'name': 'pm25 µg/m³',
     'parameter': {'id': 2,
      'name': 'pm25',
      'units': 'µg/m³',
      'displayName': 'PM2.5'}}],
   'coordinates': {'latitude': 31.560078, 'longitude': 74.33589},
   'licenses': None,
   'bounds': [74.33589, 31.560078, 74.33589, 31.560078],
   'distance': 4903.77691267,
   'datetimeFirst': {'utc': '2019-05-22T22:00:00Z',
    'local': '2019-05-23T03:00:00+05:00'},
   'datetimeLast': {'utc': '2025-03-04T12:00:00Z',
    'loc

In [24]:
# =========================
# CELL 5: Fetch OpenAQ Locations for Lahore
# =========================
locations_url = "https://api.openaq.org/v3/locations"

params = {
    "coordinates": f"{lat},{lon}",
    "radius": 25000,   # ✅ FIXED (max allowed)
    "parameters_id": 2,
    "limit": 100
}

response = requests.get(locations_url, params=params, headers=headers)

print("Status code:", response.status_code)
print(response.text[:500])

# safe parsing
if response.status_code != 200:
    raise Exception("API error, fix this before continuing")

openaq_locations = response.json()

Status code: 200
{"meta":{"name":"openaq-api","website":"/","page":1,"limit":100,"found":80},"results":[{"id":8664,"name":"US Diplomatic Post: Lahore","locality":"Lahore","timezone":"Asia/Karachi","country":{"id":109,"code":"PK","name":"Pakistan"},"owner":{"id":4,"name":"Unknown Governmental Organization"},"provider":{"id":265,"name":"StateAir Lahore"},"isMobile":false,"isMonitor":true,"instruments":[{"id":2,"name":"Government Monitor"}],"sensors":[{"id":25135,"name":"pm25 µg/m³","parameter":{"id":2,"name":"pm25


In [25]:
# =========================
# CELL 6: Display All Retrieved Locations
# =========================
locations = openaq_locations.get("results", [])

print("Total PM2.5 locations found:", len(locations))

for loc in locations:
    print("Location ID:", loc["id"])
    print("Name:", loc.get("name"))
    print("Locality:", loc.get("locality"))
    print("Country:", loc.get("country"))
    print("Parameters:", [p.get("name") for p in loc.get("parameters", [])])
    print("-" * 50)

Total PM2.5 locations found: 80
Location ID: 8664
Name: US Diplomatic Post: Lahore
Locality: Lahore
Country: {'id': 109, 'code': 'PK', 'name': 'Pakistan'}
Parameters: []
--------------------------------------------------
Location ID: 1894641
Name: Lahore
Locality: None
Country: {'id': 109, 'code': 'PK', 'name': 'Pakistan'}
Parameters: []
--------------------------------------------------
Location ID: 4515157
Name: Barki, Lahore
Locality: None
Country: {'id': 109, 'code': 'PK', 'name': 'Pakistan'}
Parameters: []
--------------------------------------------------
Location ID: 4527035
Name: Civil Secretariat
Locality: None
Country: {'id': 109, 'code': 'PK', 'name': 'Pakistan'}
Parameters: []
--------------------------------------------------
Location ID: 4527173
Name: Learning Alliance Intl. DHA
Locality: None
Country: {'id': 109, 'code': 'PK', 'name': 'Pakistan'}
Parameters: []
--------------------------------------------------
Location ID: 4527306
Name: FOOT & MOUTH DISEASE RESEARCH CEN

In [26]:
# =========================
# CELL 7: Find a Location with PM2.5 Sensor
# =========================

pm25_location_id = None
pm25_sensor_id = None
selected_location_name = None

for loc in locations:
    loc_id = loc["id"]
    loc_name = loc.get("name")

    sensors_url = f"https://api.openaq.org/v3/locations/{loc_id}/sensors"
    response = requests.get(sensors_url, headers=headers)

    # Skip if request fails
    if response.status_code != 200:
        continue

    sensors_data = response.json()
    sensors = sensors_data.get("results", [])

    for sensor in sensors:
        param_name = sensor["parameter"]["name"].lower()

        # Check for PM2.5
        if "pm2.5" in param_name or "pm25" in param_name:
            pm25_location_id = loc_id
            pm25_sensor_id = sensor["id"]
            selected_location_name = loc_name

            print("FOUND PM2.5 SENSOR")
            print("Location:", selected_location_name)
            print("Location ID:", pm25_location_id)
            print("Sensor ID:", pm25_sensor_id)
            print("Parameter:", sensor["parameter"]["name"])
            break

    if pm25_sensor_id is not None:
        break

# Final check
if pm25_sensor_id is None:
    print("No PM2.5 sensor found in these locations.")

FOUND PM2.5 SENSOR
Location: US Diplomatic Post: Lahore
Location ID: 8664
Sensor ID: 25135
Parameter: pm25


In [27]:
# =========================
# CELL 8: Fetch PM2.5 Data from Selected Sensor
# =========================

measurements_url = f"https://api.openaq.org/v3/sensors/{pm25_sensor_id}/hours"

params = {
    "datetime_from": "2022-01-01T00:00:00Z",
    "datetime_to": "2023-12-31T23:59:59Z",
    "limit": 1000,
    "page": 1
}

all_results = []

while True:
    response = requests.get(measurements_url, params=params, headers=headers)

    print("Page:", params["page"], "Status:", response.status_code)

    if response.status_code != 200:
        print(response.text[:500])
        break

    data = response.json()
    batch = data.get("results", [])

    print("Rows fetched:", len(batch))

    if len(batch) == 0:
        break

    all_results.extend(batch)
    params["page"] += 1

    # Safety stop
    if params["page"] > 30:
        break

# Convert to DataFrame
air = pd.json_normalize(all_results)

air.head()

Page: 1 Status: 200
Rows fetched: 1000
Page: 2 Status: 200
Rows fetched: 1000
Page: 3 Status: 200
Rows fetched: 1000
Page: 4 Status: 200
Rows fetched: 1000
Page: 5 Status: 200
Rows fetched: 1000
Page: 6 Status: 200
Rows fetched: 1000
Page: 7 Status: 200
Rows fetched: 1000
Page: 8 Status: 200
Rows fetched: 1000
Page: 9 Status: 200
Rows fetched: 1000
Page: 10 Status: 200
Rows fetched: 1000
Page: 11 Status: 200
Rows fetched: 1000
Page: 12 Status: 200
Rows fetched: 1000
Page: 13 Status: 200
Rows fetched: 1000
Page: 14 Status: 200
Rows fetched: 1000
Page: 15 Status: 200
Rows fetched: 1000
Page: 16 Status: 200
Rows fetched: 900
Page: 17 Status: 200
Rows fetched: 0


,value,coordinates,flagInfo.hasFlags,parameter.id,parameter.name,parameter.units,parameter.displayName,period.label,period.interval,period.datetimeFrom.utc,...,coverage.expectedCount,coverage.expectedInterval,coverage.observedCount,coverage.observedInterval,coverage.percentComplete,coverage.percentCoverage,coverage.datetimeFrom.utc,coverage.datetimeFrom.local,coverage.datetimeTo.utc,coverage.datetimeTo.local
0,369.0,None,False,2,pm25,µg/m³,None,1hour,01:00:00,2022-01-01T00:00:00Z,...,1,01:00:00,1,01:00:00,100.0,100.0,2022-01-01T00:00:00Z,2022-01-01T05:00:00+05:00,2022-01-01T01:00:00Z,2022-01-01T06:00:00+05:00
1,337.0,None,False,2,pm25,µg/m³,None,1hour,01:00:00,2022-01-01T01:00:00Z,...,1,01:00:00,1,01:00:00,100.0,100.0,2022-01-01T01:00:00Z,2022-01-01T06:00:00+05:00,2022-01-01T02:00:00Z,2022-01-01T07:00:00+05:00
2,348.0,None,False,2,pm25,µg/m³,None,1hour,01:00:00,2022-01-01T02:00:00Z,...,1,01:00:00,1,01:00:00,100.0,100.0,2022-01-01T02:00:00Z,2022-01-01T07:00:00+05:00,2022-01-01T03:00:00Z,2022-01-01T08:00:00+05:00
3,403.0,None,False,2,pm25,µg/m³,None,1hour,01:00:00,2022-01-01T03:00:00Z,...,1,01:00:00,1,01:00:00,100.0,100.0,2022-01-01T03:00:00Z,2022-01-01T08:00:00+05:00,2022-01-01T04:00:00Z,2022-01-01T09:00:00+05:00
4,531.0,None,False,2,pm25,µg/m³,None,1hour,01:00:00,2022-01-01T04:00:00Z,...,1,01:00:00,1,01:00:00,100.0,100.0,2022-01-01T04:00:00Z,2022-01-01T09:00:00+05:00,2022-01-01T05:00:00Z,2022-01-01T10:00:00+05:00


In [28]:
# =========================
# CELL 9: Inspect Data Columns
# =========================

print("Columns in dataset:")
print(air.columns)

Columns in dataset:
Index(['value', 'coordinates', 'flagInfo.hasFlags', 'parameter.id',
       'parameter.name', 'parameter.units', 'parameter.displayName',
       'period.label', 'period.interval', 'period.datetimeFrom.utc',
       'period.datetimeFrom.local', 'period.datetimeTo.utc',
       'period.datetimeTo.local', 'summary.min', 'summary.q02', 'summary.q25',
       'summary.median', 'summary.q75', 'summary.q98', 'summary.max',
       'summary.avg', 'summary.sd', 'coverage.expectedCount',
       'coverage.expectedInterval', 'coverage.observedCount',
       'coverage.observedInterval', 'coverage.percentComplete',
       'coverage.percentCoverage', 'coverage.datetimeFrom.utc',
       'coverage.datetimeFrom.local', 'coverage.datetimeTo.utc',
       'coverage.datetimeTo.local'],
      dtype='object')


In [31]:
# =========================
# CELL 10: Clean PM2.5 Data (Convert Hourly to Daily Average)
# =========================
#We converted hourly pollution readings into daily averages to align with meteorological data and reduce noise in the time series.

# Step 1: Keep only required columns
air_clean = air[["period.datetimeFrom.utc", "value"]].copy()

# Step 2: Rename columns
air_clean.rename(columns={
    "period.datetimeFrom.utc": "datetime",
    "value": "pm25"
}, inplace=True)

# Step 3: Convert to datetime format
air_clean["datetime"] = pd.to_datetime(air_clean["datetime"])

# Step 4: Extract date only (remove time)
air_clean["date"] = air_clean["datetime"].dt.date

# Step 5: Convert date back to datetime (for merging later)
air_clean["date"] = pd.to_datetime(air_clean["date"])

# Step 6: Group by date and take daily average
air_daily = air_clean.groupby("date")["pm25"].mean().reset_index()

# Preview
air_daily.head()

,date,pm25
0,2022-01-01,309.166667
1,2022-01-02,301.625000
2,2022-01-03,180.791667
3,2022-01-04,124.250000
4,2022-01-05,88.625000


In [33]:
# =========================
# CELL 11: Verify Clean PM2.5 Data
# =========================

print("Shape:", air_daily.shape)
print("Date range:")
print(air_daily["date"].min(), "to", air_daily["date"].max())

print("\nMissing values:")
print(air_daily.isnull().sum())

print("\nSummary statistics:")
print(air_daily.describe())

Shape: (666, 2)
Date range:
2022-01-01 00:00:00 to 2023-12-31 00:00:00

Missing values:
date      0
pm25    142
dtype: int64

Summary statistics:
                                date        pm25
count                            666  524.000000
mean   2022-12-30 22:22:42.162162176  125.191105
min              2022-01-01 00:00:00    2.000000
25%              2022-06-22 06:00:00   50.031250
50%              2022-12-11 12:00:00   89.719697
75%              2023-07-17 18:00:00  173.166667
max              2023-12-31 00:00:00  491.000000
std                              NaN   98.470307


In [35]:
# =========================
# CELL 12: Handle Missing PM2.5 Values
# =========================
# We handled missing PM2.5 values using linear interpolation to preserve temporal trends without introducing bias.

# Sort by date (important)
air_daily = air_daily.sort_values("date")

# Fill missing values using interpolation
air_daily["pm25"] = air_daily["pm25"].interpolate(method="linear")

# Check again
print("Missing values after fix:")
print(air_daily.isnull().sum())

air_daily.head()

Missing values after fix:
date    0
pm25    0
dtype: int64


,date,pm25
0,2022-01-01,309.166667
1,2022-01-02,301.625000
2,2022-01-03,180.791667
3,2022-01-04,124.250000
4,2022-01-05,88.625000


In [36]:
# =========================
# CELL 13: Check NASA Weather Data
# =========================

print("Weather shape:", weather.shape)
print("Weather columns:", weather.columns)

print("\nDate range:")
print(weather["date"].min(), "to", weather["date"].max())

print("\nMissing values:")
print(weather.isnull().sum())

weather.head()

Weather shape: (730, 4)
Weather columns: Index(['date', 'T2M', 'RH2M', 'WS2M'], dtype='object')

Date range:
2022-01-01 00:00:00 to 2023-12-31 00:00:00

Missing values:
date    0
T2M     0
RH2M    0
WS2M    0
dtype: int64


,date,T2M,RH2M,WS2M
0,2022-01-01,12.42,44.13,1.09
1,2022-01-02,12.92,45.62,1.37
2,2022-01-03,13.75,52.70,1.28
3,2022-01-04,12.59,69.26,1.51
4,2022-01-05,12.47,90.84,0.87


In [37]:
# =========================
# CELL 14: Merge PM2.5 Data with NASA Weather Data
# =========================

# Merge both datasets using the common date column
df = pd.merge(air_daily, weather, on="date", how="inner")

# Preview merged dataset
print("Merged dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

df.head()

Merged dataset shape: (666, 5)

Missing values:
date    0
pm25    0
T2M     0
RH2M    0
WS2M    0
dtype: int64


,date,pm25,T2M,RH2M,WS2M
0,2022-01-01,309.166667,12.42,44.13,1.09
1,2022-01-02,301.625000,12.92,45.62,1.37
2,2022-01-03,180.791667,13.75,52.70,1.28
3,2022-01-04,124.250000,12.59,69.26,1.51
4,2022-01-05,88.625000,12.47,90.84,0.87


In [39]:
# =========================
# CELL 15: Add Time-Series Features
# =========================

# Sort by date
df = df.sort_values("date")

# Previous pollution values help predict future pollution
df["pm25_lag_1"] = df["pm25"].shift(1)     # previous day PM2.5
df["pm25_lag_7"] = df["pm25"].shift(7)     # previous week PM2.5

# Extract month for seasonal patterns
df["month"] = df["date"].dt.month

# Smog season flag: Lahore smog is usually worse in Oct-Feb
df["smog_season"] = df["month"].isin([10, 11, 12, 1, 2]).astype(int)

# Remove rows where lag values are missing
df = df.dropna().reset_index(drop=True)

print("Dataset shape after feature engineering:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

df.head()

Dataset shape after feature engineering: (659, 9)

Missing values:
date           0
pm25           0
T2M            0
RH2M           0
WS2M           0
pm25_lag_1     0
pm25_lag_7     0
month          0
smog_season    0
dtype: int64


,date,pm25,T2M,RH2M,WS2M,pm25_lag_1,pm25_lag_7,month,smog_season
0,2022-01-08,97.791667,13.58,87.86,3.15,117.375000,309.166667,1,1
1,2022-01-09,98.125000,11.25,82.62,1.15,97.791667,301.625000,1,1
2,2022-01-10,218.375000,10.94,74.28,0.94,98.125000,180.791667,1,1
3,2022-01-11,162.500000,9.84,76.73,1.38,218.375000,124.250000,1,1
4,2022-01-12,371.000000,11.16,67.58,0.99,162.500000,88.625000,1,1


In [40]:
# =========================
# CELL 16: Save Final Processed Dataset
# =========================

# Save final cleaned and feature-engineered dataset
df.to_csv("lahore_air_quality_final_dataset.csv", index=False)

print("Final dataset saved successfully!")
print("File name: lahore_air_quality_final_dataset.csv")
print("Final shape:", df.shape)

Final dataset saved successfully!
File name: lahore_air_quality_final_dataset.csv
Final shape: (659, 9)


In [41]:
# =========================
# CELL 17: Download Dataset to Your Computer
# =========================

from google.colab import files

files.download("lahore_air_quality_final_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>